In [ ]:
# Install required packages (auto-skipped if already installed)
import importlib
if importlib.util.find_spec('qiskit') is None:
    !pip install -q qiskit qiskit-aer qiskit-ibm-runtime pylatexenc
else:
    print("\u2713 Packages already installed")

# To run on real quantum hardware, uncomment and fill in your credentials:
# from qiskit_ibm_runtime import QiskitRuntimeService
# QiskitRuntimeService.save_account(
#     token="<your-api-key>",
#     instance="<your-crn>",
#     overwrite=True
# )

# Introducción a los primitivos

<Accordion>
<AccordionItem title="Versiones de paquetes">

El código de esta página fue desarrollado con los siguientes requisitos.
Se recomienda usar estas versiones o más recientes.

```
qiskit[all]~=2.3.0
qiskit-ibm-runtime~=0.43.1
```
</AccordionItem>
</Accordion>
<span id="qpu-access-patterns"></span>
## ¿Por qué Qiskit introdujo los primitivos?
De forma similar a los primeros días de las computadoras clásicas, cuando los desarrolladores tenían que manipular directamente los registros de la CPU, la interfaz inicial con las QPUs simplemente devolvía los datos crudos de la electrónica de control.
Esto no representaba un gran problema cuando las QPUs se encontraban en laboratorios y solo permitían acceso directo a investigadores.
Reconociendo que la mayoría de los desarrolladores no tenían por qué estar familiarizados con la transformación de esos datos crudos en 0s y 1s, Qiskit introdujo `backend.run`, una primera abstracción para acceder a las QPUs en la nube. Esto permitió a los desarrolladores
trabajar con un formato de datos familiar y centrarse en el panorama general.

A medida que el acceso a las QPUs se fue extendiendo y se desarrollaron más algoritmos cuánticos,
surgió de nuevo la necesidad de una abstracción de mayor nivel. En respuesta, Qiskit introdujo
la interfaz de primitivos, optimizada para dos tareas fundamentales en el desarrollo de algoritmos cuánticos:
la estimación de valores esperados (`Estimator`) y el muestreo de circuitos (`Sampler`). El objetivo es, una vez
más, ayudar a los desarrolladores a centrarse más en la innovación y menos en la conversión de datos. La interfaz de primitivos reemplaza a la interfaz `backend.run`, ya que `Sampler` ofrece el mismo acceso directo al hardware que proporcionaba `backend.run`.

## ¿Qué es un primitivo?
Los sistemas de cómputo se construyen sobre múltiples capas de abstracción. Las abstracciones te permiten centrarte en
un nivel de detalle concreto, relevante para la tarea que tienes entre manos. Cuanto más cerca estés del hardware,
menor será el nivel de abstracción que necesitas (por ejemplo, es posible que tengas que mover o manipular datos a nivel de instrucción de la CPU). Cuanto más compleja sea la tarea que quieres realizar,
mayor será el nivel de abstracción necesario (por ejemplo, podrías estar usando una biblioteca de programación para realizar
cálculos algebraicos).

En este contexto, un *primitivo* es la instrucción de procesamiento más pequeña, el bloque de construcción más simple a partir del cual
se puede crear algo útil para un nivel de abstracción determinado.

El progreso reciente en computación cuántica ha aumentado la necesidad de trabajar en niveles de abstracción más altos.
A medida que el campo avanza hacia unidades de procesamiento cuántico (QPUs) más grandes y flujos de trabajo más complejos, el foco se desplaza de la interacción con señales individuales de
qubit hacia la concepción de los dispositivos cuánticos como sistemas que realizan tareas necesarias.

Las dos tareas más comunes para las computadoras cuánticas son el muestreo de estados cuánticos y el cálculo de valores esperados.
Estas tareas motivaron el diseño de los primitivos de Qiskit: **Estimator** y **Sampler**.

- Estimator calcula los valores esperados de observables respecto a los estados preparados por circuitos cuánticos.
- Sampler muestrea el registro de salida de la ejecución de circuitos cuánticos.

En resumen, el modelo computacional introducido por los primitivos de Qiskit acerca la programación cuántica un paso más
al estado actual de la programación clásica, donde el foco está menos en los detalles del hardware y más en los resultados
que se quieren obtener.

## Definición e implementaciones de primitivos
Existen dos tipos de primitivos de Qiskit: las clases base y sus implementaciones. Los primitivos Estimator y Sampler están definidos por clases base de código abierto que residen en el SDK de Qiskit (en el módulo [`qiskit.primitives`](https://docs.quantum.ibm.com/api/qiskit/primitives)). Los proveedores (como Qiskit Runtime) pueden usar estas clases base para derivar sus propias implementaciones de Sampler y Estimator. La mayoría de los usuarios interactuará con las implementaciones de los proveedores, no con los primitivos base.

### Clases base
Los primitivos `Base` son clases abstractas que definen una interfaz común para implementar primitivos. Todas las demás clases del módulo [`qiskit.primitives`](https://docs.quantum.ibm.com/api/qiskit/primitives) heredan de estas clases base. Los desarrolladores deben usarlas si están interesados en crear su propio modelo de ejecución basado en primitivos para un proveedor específico. Estas clases también pueden ser útiles para quienes desean realizar un procesamiento muy personalizado y encuentran que las implementaciones existentes de primitivos son demasiado simples para sus necesidades. Los usuarios en general no utilizarán directamente las clases base.

[`BaseEstimatorV1`](https://docs.quantum.ibm.com/api/qiskit/qiskit.primitives.BaseEstimatorV1) y [`BaseSamplerV1`](https://docs.quantum.ibm.com/api/qiskit/qiskit.primitives.BaseSamplerV1) - Aunque los primitivos V1 siguen siendo utilizables, estas guías se centran en los primitivos V2 porque son los más recientes y se usan con mayor frecuencia.

[`BaseEstimatorV2`](https://docs.quantum.ibm.com/api/qiskit/qiskit.primitives.BaseEstimatorV2) y [`BaseSamplerV2`](https://docs.quantum.ibm.com/api/qiskit/qiskit.primitives.BaseSamplerV2) - Los primitivos de referencia de Qiskit siguen estas especificaciones de interfaz.

<span id="implementations"></span>
### Implementaciones
Todos los primitivos se crean a partir de las clases base; por lo tanto, tienen la misma estructura y uso general. Por ejemplo, el formato de entrada para todos los primitivos Estimator es el mismo. Sin embargo, existen diferencias en las implementaciones que los hacen únicos.

Estas son las implementaciones de las clases base de primitivos:

- Los [primitivos de Qiskit Runtime](/guides/qiskit-runtime-primitives), [`EstimatorV2`](https://docs.quantum.ibm.com/api/qiskit-ibm-runtime/estimator-v2) y [`SamplerV2`](https://docs.quantum.ibm.com/api/qiskit-ibm-runtime/sampler-v2), proporcionan una implementación más sofisticada (por ejemplo, incluyendo mitigación de errores) como servicio en la nube. Esta implementación de los primitivos base se utiliza para acceder al hardware de IBM Quantum&reg;.

- [`StatevectorEstimator`](https://docs.quantum.ibm.com/api/qiskit/qiskit.primitives.StatevectorEstimator) y [`StatevectorSampler`](https://docs.quantum.ibm.com/api/qiskit/qiskit.primitives.StatevectorSampler#statevectorsampler) - Implementaciones de referencia de los primitivos que utilizan el simulador integrado en Qiskit. Están construidas con el módulo [`quantum_info`](https://docs.quantum.ibm.com/api/qiskit/quantum_info#quantum-information) de Qiskit y producen resultados basados en simulaciones de vector de estado ideales. Se accede a través de Qiskit. Consulta [Simulación exacta con los primitivos de Qiskit](/guides/simulate-with-qiskit-sdk-primitives) para más detalles sobre su uso.

- [`BackendEstimatorV2`](https://docs.quantum.ibm.com/api/qiskit/qiskit.primitives.BackendEstimatorV2) y [`BackendSamplerV2`](https://docs.quantum.ibm.com/api/qiskit/qiskit.primitives.BackendSamplerV2) - Puedes usar estas clases para "envolver" cualquier recurso de computación cuántica en un primitivo. Esto te permite escribir código estilo primitivo para proveedores que aún no tienen una interfaz basada en primitivos. Estas clases se pueden usar igual que Sampler y Estimator normales, salvo que deben inicializarse con un argumento adicional `backend` para seleccionar en qué computadora cuántica ejecutar. Se accede a través de Qiskit. Consulta la guía de [primitivos de backend](/guides/get-started-with-backend-primitives) para más información.

## Opciones
Puedes pasar opciones a los primitivos para personalizarlos según tus necesidades. Aunque la interfaz del método `run()` de los primitivos es común a todas las implementaciones, sus opciones no lo son. Consulta las referencias de la API para una implementación específica de primitivo para conocer las opciones que admite.

Por ejemplo, consulta los temas [Opciones de Estimator](/guides/estimator-options) y [Opciones de Sampler](/guides/sampler-options) para conocer las opciones de los primitivos de Qiskit Runtime, o consulta las [referencias de la API de Qiskit Aer](https://qiskit.github.io/qiskit-aer/apidocs/aer_primitives.html) para las opciones de los primitivos de Qiskit Aer.

## Ventajas de los primitivos de Qiskit
Con los primitivos, los usuarios de Qiskit pueden escribir código cuántico para una QPU específica sin tener que gestionar
explícitamente cada detalle. Además, gracias a la capa adicional de abstracción, es posible que puedas acceder
con mayor facilidad a las capacidades avanzadas del hardware de un proveedor determinado. Por ejemplo, con los primitivos de Qiskit Runtime,
puedes aprovechar los últimos avances en mitigación y supresión de errores activando opciones como el [`resilience_level`](https://docs.quantum.ibm.com/api/qiskit-ibm-runtime/options-estimator-options#resilience_level) del primitivo, en lugar de implementar estas técnicas por tu cuenta.

Para los proveedores de hardware, implementar primitivos de forma nativa significa que puedes ofrecer a tus usuarios una forma más "lista para usar"
de acceder a las características de tu hardware, como técnicas avanzadas de posprocesamiento. De esta manera, resulta más sencillo para tus usuarios beneficiarse de las mejores capacidades de tu hardware.

## Próximos pasos
> **Tip:** - Comprende las [entradas y salidas de los primitivos](/guides/primitive-input-output).
> - Revisa [ejemplos](/guides/simulate-with-qiskit-sdk-primitives) detallados.
> - Practica con los primitivos completando la [lección sobre funciones de costo](/learning/courses/variational-algorithm-design/cost-functions) en IBM Quantum Learning.
> - Revisa [Crear un proveedor](/guides/create-a-provider) para aprender cómo implementar tus propios primitivos Sampler y Estimator.
> - Consulta las [referencias de la API](https://docs.quantum.ibm.com/api/qiskit/primitives).
> - Lee [Migrar a los primitivos V2](/guides/v2-primitives).
> - Aprende sobre los [primitivos de Qiskit Runtime](/guides/qiskit-runtime-primitives), que se usan para ejecutar circuitos en QPUs de IBM.